## Notebook: Run U-Net saliency analysis on extreme days
### Purpose : Visualize which input features most influence the U-Net prediction at a specific location and time.

In [ ]:
# --- Auto-reload for local package development ---
# Automatically reloads modified local modules without restarting the kernel
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np

from lightning_modelling.deter_architecture import Unet
from lightning_modelling.dataset import CustomPTDataset, create_train_test, get_seasons
from lightning_modelling.plots import  SaliencyPlotPaper
from lightning_modelling.common_path import MODELS_PATH, DATASET_PATH

## Load data

In [ ]:
# Full range of available years in the dataset
ALL_YEARS = list(range(2008, 2024))

# Years reserved for evaluation only (Leave-One-Year-Out held-out set)
HELD_OUT_YEARS = [2008, 2015, 2023]

# Training years: all years except the held-out ones (LOYO strategy)
ALL_LOYO_YEARS = [year for year in ALL_YEARS if year not in HELD_OUT_YEARS]
TRAIN_YEARS = ALL_LOYO_YEARS
TEST_YEARS = HELD_OUT_YEARS

# Path to the pre-fitted scaler (fitted on the train years dataset)
SCALER_PATH = os.path.join(DATASET_PATH, "scaler", "scaler_full.pkl")

print("Creating the dataset..............")
# Only the test split is needed here; train/val outputs are discarded
_, _, TEST_DATASET = create_train_test(DATASET_PATH, TRAIN_YEARS, TEST_YEARS, scaler_path=SCALER_PATH)
TEST_DATASET.metadata_csv["year"] = pd.to_datetime(TEST_DATASET.metadata_csv["date"]).dt.year

# Load pre-computed extreme days (days with most lightning flashes observed)
extreme_days = pd.read_csv(DATASET_PATH / "extreme_days_top_0.05.csv")  

# Keep only extreme days that fall within the test years
all_extremes_metadata = TEST_DATASET.metadata_csv[TEST_DATASET.metadata_csv["date"].isin(extreme_days["date"])]
test_extremes = all_extremes_metadata[all_extremes_metadata["year"].isin(TEST_YEARS)]
test_extremes_ids = test_extremes["id"].values

# Build a dataset and dataloader restricted to extreme days only
TEST_EXTREMES_DATASET = CustomPTDataset(root_dir=DATASET_PATH, sample_ids=test_extremes_ids, scaler_path=SCALER_PATH)
TEST_EXTREMES_LOADER = DataLoader(TEST_EXTREMES_DATASET, batch_size=1, shuffle=False)

# Retrieve season labels for all years, then filter to extreme day indices
# (used for season-stratified metrics)
ALL_DAYS_SEASONS = np.array(get_seasons(ALL_YEARS))
EXTREMES_SEASONS = ALL_DAYS_SEASONS[test_extremes_ids]

## Load models

In [ ]:
# U-Net hyperparameters
CHANNELS = [16, 32, 64]
NUM_RESIDUAL_LAYERS = 2
RECALIBRATION = 'platt_scaling'

# Initialize unet
unet = Unet(
    channels = CHANNELS,
    num_residual_layers = NUM_RESIDUAL_LAYERS,
    name = "unet",
    recalibration_method = RECALIBRATION,
)

unet.load_state_dict(torch.load(MODELS_PATH / 'unet.pth', map_location=torch.device('cpu')))
unet.eval()

## Saliency plot

In [ ]:
# Preview the extreme day metadata
test_extremes.head()

In [ ]:
# Select a specific extreme event by its row-number in the filtered metadata (starting at 0)
# User can change this number to visualize different days
event_number = 1
day = test_extremes.iloc[event_number]["date"]

# Use GPU if available, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Retrieve the tensor corresponding to the event (24 hourly timesteps) for the selected event
extremes_batch = TEST_DATASET[event_number]

In [ ]:
# Target location and hour for the saliency analysis
# User can change these values to visualize different locations and hours 
lat = 40
lon = 16
hour = 18
event_time = pd.to_datetime(day) + pd.Timedelta(f"{hour}:00:00")

# Slice the input to a single hour; exclude the last channel (observation target)
data = extremes_batch[hour:hour+1, :-1]

In [ ]:

saliency = SaliencyPlotPaper(
    data,   # Shape passed: (1, C, H, W) — one time step, all feature channels
    unet,
    TEST_DATASET.metadata_json,
    lat,
    lon,
    f"Saliency map on {event_time.strftime('%Y-%m-%d')} at {event_time.strftime('%H:%M')} at location {lat}N, {lon}E",
    device,
    save_path=None,
    file_name=None
)